# RFM Customer Analysis

RFM analysis segments customers using three measures:

- **Recency**: how recently a customer purchased.
- **Frequency**: how often a customer purchased, measured here by distinct qualifying invoices.
- **Monetary value**: how much a customer spent on qualifying sales.

This helps an ecommerce business identify loyal customers, high-value customers, new buyers, inactive customers, and customers at risk of churning. Customer IDs are required, so transactions without a usable `CustomerID` are excluded from customer-level RFM analysis. The reference date is the latest date in this historical dataset.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_excel("Online Retail.xlsx")

clean_df = df.copy()
clean_df["InvoiceNo"] = clean_df["InvoiceNo"].astype(str).str.strip()
clean_df["StockCode"] = clean_df["StockCode"].astype(str).str.strip()
clean_df["InvoiceDate"] = pd.to_datetime(clean_df["InvoiceDate"], errors="coerce")
clean_df["Quantity"] = pd.to_numeric(clean_df["Quantity"], errors="coerce")
clean_df["UnitPrice"] = pd.to_numeric(clean_df["UnitPrice"], errors="coerce")
clean_df = clean_df.drop_duplicates().copy()
clean_df["IsCancelled"] = clean_df["InvoiceNo"].str.startswith("C")
clean_df["IsReturned"] = clean_df["Quantity"] < 0

required_columns = [
    "InvoiceNo", "StockCode", "Description", "Quantity",
    "InvoiceDate", "UnitPrice", "CustomerID", "Country"
]
clean_df = clean_df.dropna(subset=required_columns).copy()

non_product_codes = {
    "POST", "DOT", "M", "MANUAL", "D", "DISCOUNT",
    "AMAZONFEE", "BANK CHARGES", "CRUK", "S"
}
clean_df["StockCodeUpper"] = clean_df["StockCode"].str.upper()
clean_df = clean_df[
    (~clean_df["IsCancelled"]) &
    (~clean_df["IsReturned"]) &
    (clean_df["Quantity"] > 0) &
    (clean_df["UnitPrice"] > 0) &
    (~clean_df["StockCodeUpper"].isin(non_product_codes))
].copy()
clean_df["Revenue"] = clean_df["Quantity"] * clean_df["UnitPrice"]

print(f"Qualifying sales rows: {len(clean_df):,}")
print(f"Qualifying revenue: GBP {clean_df['Revenue'].sum():,.2f}")
print(f"Customers available for RFM: {clean_df['CustomerID'].nunique():,}")

In [ ]:
reference_date = clean_df["InvoiceDate"].max().normalize()

customer_rfm = (
    clean_df.groupby("CustomerID", as_index=False)
    .agg(
        LastPurchase=("InvoiceDate", "max"),
        Frequency=("InvoiceNo", "nunique"),
        MonetaryValue=("Revenue", "sum"),
        ItemsPurchased=("Quantity", "sum"),
    )
)
customer_rfm["Recency"] = (
    reference_date - customer_rfm["LastPurchase"].dt.normalize()
).dt.days

customer_rfm = customer_rfm[
    [
        "CustomerID", "Recency", "Frequency", "MonetaryValue",
        "ItemsPurchased", "LastPurchase",
    ]
]

display(customer_rfm.head())
print(f"RFM reference date: {reference_date.date()}")
display(customer_rfm[["Recency", "Frequency", "MonetaryValue"]].describe().round(2))

In [ ]:
# Higher recency scores mean more recent purchases; higher frequency and monetary scores mean higher values.
customer_rfm["R_Score"] = pd.qcut(
    customer_rfm["Recency"].rank(method="first"),
    5,
    labels=[5, 4, 3, 2, 1],
).astype(int)
customer_rfm["F_Score"] = pd.qcut(
    customer_rfm["Frequency"].rank(method="first"),
    5,
    labels=[1, 2, 3, 4, 5],
).astype(int)
customer_rfm["M_Score"] = pd.qcut(
    customer_rfm["MonetaryValue"].rank(method="first"),
    5,
    labels=[1, 2, 3, 4, 5],
).astype(int)
customer_rfm["RFM_Score"] = (
    customer_rfm["R_Score"].astype(str)
    + customer_rfm["F_Score"].astype(str)
    + customer_rfm["M_Score"].astype(str)
)

def assign_segment(row):
    if row["R_Score"] >= 4 and row["F_Score"] >= 4 and row["M_Score"] >= 4:
        return "Champions"
    if row["R_Score"] >= 3 and row["F_Score"] >= 4:
        return "Loyal Customers"
    if row["M_Score"] >= 4 and row["F_Score"] <= 3:
        return "Big Spenders"
    if row["R_Score"] >= 4 and row["F_Score"] <= 2:
        return "New Customers"
    if row["R_Score"] <= 2 and row["M_Score"] >= 3:
        return "At Risk"
    if row["R_Score"] <= 2 and row["F_Score"] <= 2:
        return "Lost Customers"
    return "Developing Customers"

customer_rfm["Segment"] = customer_rfm.apply(assign_segment, axis=1)
display(customer_rfm.head(10))

In [ ]:
segment_summary = (
    customer_rfm.groupby("Segment", as_index=False)
    .agg(
        Customers=("CustomerID", "nunique"),
        Revenue=("MonetaryValue", "sum"),
        AverageCustomerValue=("MonetaryValue", "mean"),
        AverageFrequency=("Frequency", "mean"),
        AverageRecency=("Recency", "mean"),
    )
    .sort_values("Revenue", ascending=False)
)
segment_summary["RevenueSharePct"] = (
    segment_summary["Revenue"] / segment_summary["Revenue"].sum() * 100
)

print("RFM SEGMENT SUMMARY")
display(
    segment_summary.round(
        {"Revenue": 2, "AverageCustomerValue": 2, "AverageFrequency": 2, "AverageRecency": 2, "RevenueSharePct": 2}
    )
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

segment_counts = segment_summary.sort_values("Customers", ascending=False)
axes[0].bar(segment_counts["Segment"], segment_counts["Customers"])
axes[0].set_title("Customers by RFM Segment")
axes[0].set_ylabel("Number of Customers")
axes[0].tick_params(axis="x", rotation=45)

segment_revenue = segment_summary.sort_values("Revenue", ascending=False)
axes[1].bar(segment_revenue["Segment"], segment_revenue["Revenue"])
axes[1].set_title("Revenue by RFM Segment")
axes[1].set_ylabel("Revenue (GBP)")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()